# Mammogram Classification with XAI (Grad-CAM)
## Target: 3-Class Classification (Normal, Benign, Malignant)
## Dataset: King Abdulaziz University Mammogram Dataset

**Mapping Strategy:**
- BI-RADS 1 → Normal (Class 0)
- BI-RADS 3 → Benign (Class 1)
- BI-RADS 4 & 5 → Malignant (Class 2) [Combined for clinical relevance as per PMC5274695]

In [ ]:
# 1. Imports & Setup
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models, applications, preprocessing
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# 2. Data Preprocessing with CLAHE (as recommended in PMC5274695)
def preprocess_mammogram(image_path, target_size=(512, 512)):
    # Read image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    # Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(img)
    
    # Resize to target size
    img_resized = cv2.resize(img_clahe, target_size)
    
    # Normalize to [0, 1]
    img_normalized = img_resized.astype(np.float32) / 255.0
    
    # Stack to 3 channels for CNN compatibility (e.g., EfficientNet)
    img_3ch = np.stack((img_normalized,) * 3, axis=-1)
    return img_3ch

In [ ]:
# 3. Data Loading and Mapping
DATA_DIR = '/path/to/king-abdulaziz-university-mammogram-dataset' # Update this path

image_paths = []
labels = []

# Mapping BI-RADS to 3 classes: 0: Normal, 1: Benign, 2: Malignant
class_mapping = {
    'BIRAD1': 0,      # Normal
    'Birad3': 1,      # Benign
    'Birad4': 2,      # Malignant
    'Birad5': 2       # Malignant
}

class_names = ['Normal (BI-RADS 1)', 'Benign (BI-RADS 3)', 'Malignant (BI-RADS 4/5)']

for folder_name, label in class_mapping.items():
    folder_path = os.path.join(DATA_DIR, folder_name)
    if not os.path.exists(folder_path):
        print(f"Warning: Folder {folder_path} not found.")
        continue
    
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff')):
            image_paths.append(os.path.join(folder_path, filename))
            labels.append(label)

print(f"Total images loaded: {len(image_paths)}")

# Split data
X_train, X_temp, y_train, y_temp = train_test_split(image_paths, labels, test_size=0.3, random_state=42, stratify=labels)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# 4. Custom Data Generator (Memory Efficient)
def data_generator(paths, labels, batch_size=16, target_size=(512, 512), augment=False):
    while True:
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            
            images = []
            valid_labels = []
            
            for path, label in zip(batch_paths, batch_labels):
                img = preprocess_mammogram(path, target_size)
                if img is not None:
                    if augment:
                        # Basic augmentation: random flip and rotation
                        img = tf.image.random_flip_left_right(img)
                        img = tf.image.random_flip_up_down(img)
                        img = tf.image.rot90(img, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))
                    images.append(img)
                    valid_labels.append(label)
            
            if len(images) > 0:
                yield np.array(images), tf.keras.utils.to_categorical(valid_labels, num_classes=3)

In [ ]:
# 5. Model Building (Transfer Learning with EfficientNetB0)
IMG_SIZE = (512, 512)
BATCH_SIZE = 16

base_model = applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(*IMG_SIZE, 3),
    pooling='avg'
)
base_model.trainable = False  # Freeze initially

inputs = layers.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(3, activation='softmax')(x)

model = models.Model(inputs, outputs)

# Calculate class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class Weights: {class_weight_dict}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(multi_class='ovr', name='auc')]
)
model.summary()

In [ ]:
# 6. Training
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_mammogram_model.h5', monitor='val_auc', save_best_only=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
]

train_gen = data_generator(X_train, y_train, batch_size=BATCH_SIZE, augment=True)
val_gen = data_generator(X_val, y_val, batch_size=BATCH_SIZE, augment=False)

history = model.fit(
    train_gen,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    validation_data=val_gen,
    validation_steps=len(X_val) // BATCH_SIZE,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

In [ ]:
# 7. Evaluation
test_gen = data_generator(X_test, y_test, batch_size=BATCH_SIZE, augment=False)
y_pred_probs = model.predict(test_gen, steps=len(X_test) // BATCH_SIZE + 1)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = y_test[:len(y_pred_classes)] # Adjust for last batch

print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))

cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# 8. XAI Integration: Grad-CAM Implementation
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # First, we create a model that maps the input image to the activations
    # of the last conv layer as well as the output predictions
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, model, last_conv_layer_name='top_conv'):
    img = preprocess_mammogram(img_path, IMG_SIZE)
    img_array = np.expand_dims(img, axis=0)
    
    preds = model.predict(img_array)
    pred_class = np.argmax(preds[0])
    confidence = preds[0][pred_class]
    
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_class)
    
    # Rescale heatmap to 0-255 and apply colormap
    heatmap = np.uint8(255 * heatmap)
    jet = plt.colormaps['jet']
    heatmap_colored = jet(heatmap)[..., :3] * 255
    
    # Read original image for overlay
    original_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    original_img_resized = cv2.resize(original_img, IMG_SIZE)
    original_img_3ch = np.stack((original_img_resized,) * 3, axis=-1)
    
    # Superimpose
    superimposed_img = heatmap_colored * 0.4 + original_img_3ch * 0.6
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(original_img_resized, cmap='gray')
    axes[0].set_title('Original (CLAHE)')
    axes[0].axis('off')
    
    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap')
    axes[1].axis('off')
    
    axes[2].imshow(superimposed_img)
    axes[2].set_title(f'Prediction: {class_names[pred_class]}\nConfidence: {confidence:.2f}')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Test Grad-CAM on a few samples
for i in range(3):
    display_gradcam(X_test[i], model, last_conv_layer_name='top_conv')